# 14.5 - Reflection

Status: VERIFIED

## What Are We Solving?
Reflection is a self-evaluation step where the agent reviews its own output before finalizing. LLMs can produce confident but wrong outputs — reflection catches errors.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Reflect and Revise Pattern

In [2]:
def reflect_and_revise(task: str, draft: str, max_revisions: int = 2) -> dict:
    """Have the LLM critique and revise its own output."""
    current = draft
    revisions = []
    
    for revision in range(max_revisions):
        # Critique
        critique_response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": (
                    "Review this draft response against the task. "
                    "List specific issues: missing information, errors, unclear parts. "
                    "If no issues remain, respond with just: APPROVED"
                )},
                {"role": "user", "content": f"Task: {task}\n\nDraft: {current}"}
            ]
        )
        critique = critique_response.choices[0].message.content
        
        if "APPROVED" in critique:
            revisions.append({"round": revision + 1, "action": "approved"})
            print(f"  Round {revision + 1}: APPROVED")
            return {"final": current, "revisions": revisions}
        
        # Revise
        revision_response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Revise the response to fix the listed issues. Be concise."},
                {"role": "user", "content": f"Task: {task}\n\nCritique: {critique}\n\nDraft: {current}"}
            ]
        )
        current = revision_response.choices[0].message.content
        revisions.append({"round": revision + 1, "action": "revised", "issues": critique[:100]})
        print(f"  Round {revision + 1}: Revised")
    
    return {"final": current, "revisions": revisions}

# Test
result = reflect_and_revise(
    "Explain overfitting",
    "Overfitting is bad."
)
print(f"\nFinal ({len(result['revisions'])} revisions): {result['final'][:200]}")

  Round 1: Revised


  Round 2: APPROVED

Final (2 revisions): Overfitting occurs when a model learns noise in its training data instead of the underlying pattern, typically due to excessive complexity or insufficient data. This results in poor generalization per


In [3]:
# Verification
assert result["final"] is not None
assert len(result["revisions"]) > 0
print("VERIFICATION PASSED: Phase 14.5 complete")

VERIFICATION PASSED: Phase 14.5 complete
